**Item Recommendation System**


This project implements a content-based item recommendation system designed to suggest similar products to a user. The system uses a lightweight k-Nearest Neighbors (k-NN) model trained on a preprocessed dataset. Below is a detailed explanation of the process from data preprocessing to backend integration.

In [30]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors
import joblib
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors

In [2]:
# Step 1: Load Individual Datasets
# Replace these paths with the actual paths to your CSV files
aldi_data = pd.read_csv('Aldi_Cleaned.csv')
coles_data = pd.read_csv('Coles_Cleaned.csv')
iga_data = pd.read_csv('IGA_Cleaned.csv')
woolworths_data = pd.read_csv('Woolworths_Cleaned.csv')

In [10]:
# Step 1: Remove Duplicate Columns
def remove_duplicate_columns(df):
    return df.loc[:, ~df.columns.duplicated()]

aldi_data = remove_duplicate_columns(aldi_data)
coles_data = remove_duplicate_columns(coles_data)
iga_data = remove_duplicate_columns(iga_data)
woolworths_data = remove_duplicate_columns(woolworths_data)

In [13]:
# Step 2: Standardize Column Names
def standardize_columns(df, store_name):
    """Standardize column names for each dataset."""
    df = df.rename(columns={
        'Product Name': 'Product',
        'Brand_Product_Size': 'Product',  # Align alternative column names
        'Price': f'Price_{store_name}',
        'Unit Price': f'ppu_{store_name}',
        'Category': f'Category_{store_name}',
        'COL Price': f'Price_{store_name}',
        'COL ppu': f'ppu_{store_name}',
        'COL Category': f'Category_{store_name}',
        'IGA Price': f'Price_{store_name}',
        'IGA ppu': f'ppu_{store_name}',
        'IGA Category': f'Category_{store_name}',
        'WOW Price': f'Price_{store_name}',
        'WOW ppu': f'ppu_{store_name}',
        'WOW Category': f'Category_{store_name}',
        'Product URL': f'Product_URL_{store_name}',
        'URL': f'Product_URL_{store_name}'
    }, errors='ignore')  # Ignore missing columns

    # Select relevant columns for merging
    valid_columns = [col for col in [
        'Product', f'Price_{store_name}', f'ppu_{store_name}',
        f'Category_{store_name}', f'Product_URL_{store_name}'
    ] if col in df.columns]

    return df[valid_columns]

aldi_data = standardize_columns(aldi_data, 'Aldi')
coles_data = standardize_columns(coles_data, 'Coles')
iga_data = standardize_columns(iga_data, 'IGA')
woolworths_data = standardize_columns(woolworths_data, 'Woolworths')

In [14]:
# Step 3: Check for Null or Duplicate Product Columns
print("Checking for duplicates or nulls in Product columns:")
print("Aldi Data Null Products:", aldi_data['Product'].isna().sum())
print("Coles Data Null Products:", coles_data['Product'].isna().sum())
print("IGA Data Null Products:", iga_data['Product'].isna().sum())
print("Woolworths Data Null Products:", woolworths_data['Product'].isna().sum())

Checking for duplicates or nulls in Product columns:
Aldi Data Null Products: 0
Coles Data Null Products: 0
IGA Data Null Products: 0
Woolworths Data Null Products: 0


In [15]:
# Drop rows with null Product values
aldi_data = aldi_data.dropna(subset=['Product'])
coles_data = coles_data.dropna(subset=['Product'])
iga_data = iga_data.dropna(subset=['Product'])
woolworths_data = woolworths_data.dropna(subset=['Product'])

In [16]:
# Ensure unique Products
aldi_data = aldi_data.drop_duplicates(subset=['Product'])
coles_data = coles_data.drop_duplicates(subset=['Product'])
iga_data = iga_data.drop_duplicates(subset=['Product'])
woolworths_data = woolworths_data.drop_duplicates(subset=['Product'])

In [17]:
# Step 4: Merge Datasets
merged_data = pd.merge(aldi_data, coles_data, on='Product', how='outer', suffixes=('_Aldi', '_Coles'))
merged_data = pd.merge(merged_data, iga_data, on='Product', how='outer', suffixes=('', '_IGA'))
merged_data = pd.merge(merged_data, woolworths_data, on='Product', how='outer', suffixes=('', '_Woolworths'))

In [33]:
# Define the path where the file will be saved
save_path = "merged_data.csv"

# Save the DataFrame
merged_data.to_csv(save_path, index=False)

# Confirm the save action
print(f"merged_data.csv has been saved in the models directory at: {save_path}")

merged_data.csv has been saved in the models directory at: merged_data.csv


**Data Preprocessing**

**Standardization:**

Unified column names across all datasets for consistency.
Ensured that each dataset includes the following columns: Product, Price, Category, and Product_URL.

**Price Normalization:**

Converted non-numeric values (e.g., "$1.23 per 100g") to numeric values.
Filled missing prices with the average price across stores for each product.

**Category Encoding:**

Combined categories from all datasets into a single column.
Used one-hot encoding to transform categories into numeric features.

**Store Availability:**

Created binary features (Aldi_Available, Coles_Available, etc.) to indicate whether a product is available in a store.


In [21]:
# Step 1: Normalize Price Columns (Convert to Numeric)
def normalize_price(price):
    """Convert price to numeric, handling non-numeric values."""
    try:
        # Extract numeric value from price, handle cases like "$1.23 per 100g"
        numeric_price = float(''.join(filter(lambda x: x.isdigit() or x == '.', str(price))))
        return numeric_price
    except:
        return np.nan

In [22]:
# Apply normalization to all price columns
merged_data['Price_Aldi'] = merged_data['Price_Aldi'].apply(normalize_price)
merged_data['Price_Coles'] = merged_data['Price_Coles'].apply(normalize_price)
merged_data['Price_IGA'] = merged_data['Price_IGA'].apply(normalize_price)
merged_data['Price_Woolworths'] = merged_data['Price_Woolworths'].apply(normalize_price)

In [23]:
# Step 2: Fill Missing Price Data with Average Price Across Stores
def compute_average_price(row):
    prices = [row['Price_Aldi'], row['Price_Coles'], row['Price_IGA'], row['Price_Woolworths']]
    prices = [p for p in prices if not pd.isna(p)]
    return np.mean(prices) if prices else np.nan

merged_data['Average_Price'] = merged_data.apply(compute_average_price, axis=1)

In [24]:
# Step 3: Verify the Results
print("Normalized and Averaged Prices:")
print(merged_data[['Product', 'Price_Aldi', 'Price_Coles', 'Price_IGA', 'Price_Woolworths', 'Average_Price']].head())

Normalized and Averaged Prices:
                                            Product  Price_Aldi  Price_Coles  \
0                 100 Plus Isotonic Drink Can 325mL         NaN          1.3   
1                 100 Plus Sport Isotonic Can 325mL         NaN          NaN   
2       1000 Hour Black Eyelash & Brow Dye Kit each         NaN         20.0   
3  1000 Hour Dark Brown Eyelash & Brow Dye Kit each         NaN         20.0   
4             1000 Hour Envious Black Lashes 1 pack         NaN          9.5   

   Price_IGA  Price_Woolworths  Average_Price  
0        NaN               NaN            1.3  
1        NaN               1.3            1.3  
2        NaN               NaN           20.0  
3        NaN               NaN           20.0  
4        NaN               NaN            9.5  


In [25]:
# Step 1: Encode Store Availability
def encode_store_availability(row):
    availability = {
        'Aldi': not pd.isna(row['Price_Aldi']),
        'Coles': not pd.isna(row['Price_Coles']),
        'IGA': not pd.isna(row['Price_IGA']),
        'Woolworths': not pd.isna(row['Price_Woolworths']),
    }
    return availability

store_availability = merged_data.apply(encode_store_availability, axis=1, result_type='expand')
store_availability.columns = ['Aldi_Available', 'Coles_Available', 'IGA_Available', 'Woolworths_Available']

In [26]:
# Step 2: Combine and One-Hot Encode Categories
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')
categories = merged_data[['Category_Aldi', 'Category_Coles', 'Category_IGA', 'Category_Woolworths']].fillna('Unknown')
categories_combined = categories.mode(axis=1)[0]  # Combine categories into a single column
categories_encoded = encoder.fit_transform(categories_combined.values.reshape(-1, 1))

c:\Users\NISHANT KHAMKAR\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [27]:
# Step 3: Normalize Average Price
scaler = StandardScaler()
average_price_scaled = scaler.fit_transform(merged_data[['Average_Price']].fillna(0))

In [28]:
# Step 4: Construct the Feature Matrix
feature_matrix = np.hstack((categories_encoded, average_price_scaled, store_availability.values))

In [29]:
# Verify Feature Matrix
print("Feature Matrix Shape:", feature_matrix.shape)
print("Sample Feature Matrix:")
print(feature_matrix[:5])

Feature Matrix Shape: (43870, 16)
Sample Feature Matrix:
[[ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.         -0.67081651
   0.          1.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.         -0.67081651
   0.          0.          0.          1.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.          1.30829934
   0.          1.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.          1.30829934
   0.          1.          0.          0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          1.          0.19703108
   0.          1.          0.          

**Model Details**

Algorithm: k-Nearest Neighbors (k-NN)

Similarity Metric: Cosine similarity

**Features Used:**

One-hot encoded product categories.

Normalized average price.

Store availability indicators.

In [31]:
# Step 1: Train k-NN Model
knn_model = NearestNeighbors(n_neighbors=10, algorithm='auto', metric='cosine')
knn_model.fit(feature_matrix)

NearestNeighbors(metric='cosine', n_neighbors=10)

In [32]:
# Step 2: Save the Model and Feature Matrix
joblib.dump(knn_model, "item_recommendation_knn_model.joblib")
np.save("feature_matrix.npy", feature_matrix)

**Recommendation Workflow**

**Input:**

A product index and the number of recommendations (k).

**Process:**

The k-NN model computes the similarity between the input product and all other products in the dataset.
Retrieves the top k most similar products based on cosine similarity.
Output:

**A list of recommended products, each including:**

Product name

Similarity score

Price

Store availability